# 01 - Data Audit and Cleaning

This notebook handles data extraction, cleaning (using mathematical and statistical imputation), validation, and structural splitting of the Indian Power Grid dataset.

In [1]:
import pandas as pd
import numpy as np
import os

# Define relative paths for reproducibility
INPUT_FILE = 'Energy and Peak DATA.xlsx'
OUTPUT_DIR = 'data/processed/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Data Extraction
Load the required sheets.

In [2]:
print(f"Loading raw data from {INPUT_FILE}...")

df_monthly = pd.read_excel(INPUT_FILE, sheet_name="psp_fytd_cumulative")
df_entity = pd.read_excel(INPUT_FILE, sheet_name="dim_entity.csv")

df_monthly.head()

Loading raw data from Energy and Peak DATA.xlsx...


,month_start,year,month,fy,fy_month_index,days_in_month,entity_id,entity_name,entity_type,region,...,demand_nm_pct,demand_nm_pct_calc,energy_req_per_day_mu,load_factor_pct,energy_carried_forward_marker,peak_carried_forward_marker,energy_partial_row,peak_partial_row,energy_source_file,peak_source_file
0,2026-01-01,2026,1,FY2025-26,10,31,all_india,All India,all_india,All India,...,0.0,0.0114,4565.2258,77.4716,0.0,0.0,0.0,0.0,Energy_January_26_Revised.pdf,Peak_January_26_Revised.pdf
1,2026-01-01,2026,1,FY2025-26,10,31,others_er,Others (ER),others,Eastern Region,...,NaN,NaN,0.1935,NaN,0.0,0.0,0.0,1.0,Energy_January_26_Revised.pdf,Peak_January_26_Revised.pdf
2,2026-01-01,2026,1,FY2025-26,10,31,eastern_region,Eastern Region,region,Eastern Region,...,0.2,0.1525,491.0323,82.0618,0.0,0.0,0.0,0.0,Energy_January_26_Revised.pdf,Peak_January_26_Revised.pdf
3,2026-01-01,2026,1,FY2025-26,10,31,andaman_nicobar,Andaman & Nicobar,state_ut,Eastern Region,...,9.1,9.0909,1.1613,78.4050,0.0,0.0,0.0,0.0,Energy_January_26_Revised.pdf,Peak_January_26_Revised.pdf
4,2026-01-01,2026,1,FY2025-26,10,31,bihar,Bihar,state_ut,Eastern Region,...,0.1,0.1224,111.4516,71.1272,0.0,0.0,0.0,0.0,Energy_January_26_Revised.pdf,Peak_January_26_Revised.pdf


## 2. Data Cleaning
We apply non-negotiable rules and advanced imputation for missing values without losing meaning. Missing values are filled using rigorous statistical measures (ffill, bfill, median) grouped by entity to prevent data leakage across regions.

In [3]:
# 1. Mathematical Imputation (Exact Relationships)

# Demand Not Met = Peak Demand - Peak Met
mask_demand_missing = df_monthly['demand_not_met_mw'].isnull() & df_monthly['peak_demand_mw'].notnull() & df_monthly['peak_met_mw'].notnull()
df_monthly.loc[mask_demand_missing, 'demand_not_met_mw'] = df_monthly['peak_demand_mw'] - df_monthly['peak_met_mw']

# Peak Met = Peak Demand - Demand Not Met
mask_met_missing = df_monthly['peak_met_mw'].isnull() & df_monthly['peak_demand_mw'].notnull() & df_monthly['demand_not_met_mw'].notnull()
df_monthly.loc[mask_met_missing, 'peak_met_mw'] = df_monthly['peak_demand_mw'] - df_monthly['demand_not_met_mw']

# Load Factor calculation
mask_lf_missing = df_monthly['load_factor_pct'].isnull() & df_monthly['energy_requirement_mu'].notnull() & (df_monthly['peak_demand_mw'] > 0)
calculated_lf = (df_monthly.loc[mask_lf_missing, 'energy_requirement_mu'] * 1000) / (df_monthly.loc[mask_lf_missing, 'peak_demand_mw'] * 24 * df_monthly.loc[mask_lf_missing, 'days_in_month']) * 100
df_monthly.loc[mask_lf_missing, 'load_factor_pct'] = calculated_lf

In [4]:
# 2. Statistical Imputation (Forward Fill, Backward Fill, Median, Mean)

# Sort by entity and time for accurate time-series filling
df_monthly['month_start'] = pd.to_datetime(df_monthly['month_start'])
df_monthly.sort_values(by=['entity_id', 'month_start'], inplace=True)

# For cumulative/time-series values, forward fill then backward fill makes sense within each entity group.
cols_to_ffill = ['energy_requirement_mu', 'energy_supplied_mu', 'energy_not_supplied_mu', 'peak_demand_mw', 'peak_met_mw', 'demand_not_met_mw']

df_monthly[cols_to_ffill] = df_monthly.groupby('entity_id')[cols_to_ffill].transform(lambda x: x.ffill().bfill())

# For percentages (like load_factor_pct), use the median of that specific entity across time.
df_monthly['load_factor_pct'] = df_monthly.groupby('entity_id')['load_factor_pct'].transform(lambda x: x.fillna(x.median()))

# If there are still missing values for an entity with NO history, fill with the global mean or 0 as a last resort.
df_monthly['load_factor_pct'] = df_monthly['load_factor_pct'].fillna(df_monthly['load_factor_pct'].mean())
df_monthly[cols_to_ffill] = df_monthly[cols_to_ffill].fillna(0)

print("Missing values rigorously handled using temporal ffill/bfill and grouped median/mean.")

Missing values rigorously handled using temporal ffill/bfill and grouped median/mean.


## 3. Data Validation
Ensure no negative metrics and no duplicate temporal records for the same entity.

In [5]:
metric_cols = ['energy_requirement_mu', 'energy_supplied_mu', 'energy_not_supplied_mu',
               'peak_demand_mw', 'peak_met_mw', 'demand_not_met_mw']

# Check No Negatives
for col in metric_cols:
    if (df_monthly[col].dropna() < 0).any():
        raise ValueError(f"Negative values found in {col}")

# Check Duplicates
if df_monthly.duplicated(subset=['entity_id', 'month_start']).any():
    raise ValueError("Duplicate records found for entity and month.")

print("Validation Passed: No negatives and no duplicates.")

Validation Passed: No negatives and no duplicates.


## 4. Structural Split (Grain Separation)
We separate aggregate rows (Regions, All-India) from granular rows (States/UTs) so ML/EDA models don't double-count them. This fixes a major flaw in previous code where Regions and All India were mixed.

In [6]:
df_state = df_monthly[df_monthly['entity_type'].isin(['state_ut', 'utility'])].copy()
df_region = df_monthly[df_monthly['entity_type'] == 'region'].copy()
df_all_india = df_monthly[df_monthly['entity_type'] == 'all_india'].copy()

# Drop administrative columns not needed for modeling
drop_cols = ['energy_carried_forward_marker', 'peak_carried_forward_marker', 
             'energy_source_file', 'peak_source_file', 'energy_partial_row', 'peak_partial_row']

for df_split in [df_state, df_region, df_all_india]:
    df_split.drop(columns=drop_cols, inplace=True, errors='ignore')

## 5. Serialization
Export final CSVs for EDA and ML.

In [7]:
dim_path = os.path.join(OUTPUT_DIR, "dim_entities.csv")
state_path = os.path.join(OUTPUT_DIR, "fact_power_state_level.csv")
region_path = os.path.join(OUTPUT_DIR, "fact_power_regional_level.csv")
all_india_path = os.path.join(OUTPUT_DIR, "fact_power_all_india.csv")

df_entity.to_csv(dim_path, index=False)
df_state.to_csv(state_path, index=False)
df_region.to_csv(region_path, index=False)
df_all_india.to_csv(all_india_path, index=False)

print("Pipeline Execution Complete.")
print(f"Exported {len(df_entity)} entities to {dim_path}")
print(f"Exported {len(df_state)} state-level records to {state_path}")
print(f"Exported {len(df_region)} regional records to {region_path}")
print(f"Exported {len(df_all_india)} all-India records to {all_india_path}")

Pipeline Execution Complete.
Exported 47 entities to data/processed/dim_entities.csv
Exported 288 state-level records to data/processed/fact_power_state_level.csv
Exported 40 regional records to data/processed/fact_power_regional_level.csv
Exported 8 all-India records to data/processed/fact_power_all_india.csv
